# Vehicle Speed Estimation using Ultralytics YOLO26

**Computer Vision Project | Object Detection, Tracking & Speed Estimation**

## 1. Project Overview

This notebook demonstrates how to estimate the speed of moving objects (such as vehicles) in a video using the Ultralytics YOLO26 Speed Estimation solution.

The system processes a video frame by frame. It detects objects, tracks their movement across consecutive frames, and estimates their speed using the video frame rate (`fps`) and a real-world scale factor (`meter_per_pixel`).

### Project Objectives
- Detect and track objects in a traffic video.
- Estimate the speed of each tracked object.
- Display the estimated speed on the processed video.
- Save the annotated output video for later review.

> **Important:** The estimated speed is not guaranteed to be exact. Accuracy depends on camera position, video quality, object tracking, frame rate, and especially the calibration of `meter_per_pixel`.


## 2. How Does Speed Estimation Work?

Speed estimation combines two main computer vision tasks:

1. **Object Detection:** YOLO identifies objects such as cars, buses, and trucks in each video frame.
2. **Object Tracking:** The tracking component assigns an ID to each object and follows it across frames.
3. **Movement Measurement:** The system measures the object's displacement in pixels over a number of frames.
4. **Speed Calculation:** Pixel displacement is converted into meters using `meter_per_pixel`, then divided by elapsed time.

The basic physical formula is:

\[
\text{Speed} = \frac{\text{Distance}}{\text{Time}}
\]

The result is converted from meters per second to kilometers per hour by multiplying by **3.6**.

### Main Parameters

| Parameter | Meaning |
|---|---|
| `fps` | Number of video frames processed per second; used to calculate elapsed time |
| `meter_per_pixel` | Real-world meters represented by one pixel in the scene |
| `max_hist` | Number of historical positions used before calculating a speed |
| `max_speed` | Upper limit for the estimated speed, in km/h |
| `classes` | Optional object-class filter |


## 3. Install and Import Required Libraries

We use:
- **Ultralytics:** Provides the YOLO model and the ready-to-use `SpeedEstimator` solution.
- **OpenCV:** Reads the input video, accesses its properties, and writes the processed video.


In [1]:
pip install opencv-python

In [2]:
pip install ultralytics

In [3]:
import numpy as np
import cv2
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt


### Code Explanation

- `import cv2` imports OpenCV, which we will use to open the video, read its frames, and save the output.
- `from ultralytics import solutions` imports the Ultralytics solutions module, which contains the `SpeedEstimator` class.

We import only the components needed for this workflow.

In [16]:
import cv2

from ultralytics import solutions

cap = cv2.VideoCapture("/content/video99.mp4")
assert cap.isOpened(), "Error reading video file"

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("speed_management.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Initialize speed estimation object
speedestimator = solutions.SpeedEstimator(
    show=True,  # display the output
    model="yolo26n.pt",  # path to the YOLO26 model file.
    fps=fps,  # adjust speed based on frame per second
    # max_speed=120,  # cap speed to a max value (km/h) to avoid outliers
    # max_hist=5,  # minimum frames object tracked before computing speed
    # meter_per_pixel=0.05,  # highly depends on the camera configuration
    # classes=[0, 2],  # estimate speed of specific classes.
     line_width=4,   #adjust the line width for bounding boxes
)

# Process video
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        print("Video frame is empty or processing is complete.")
        break

    results = speedestimator(im0)

    # print(results)  # access the output

    video_writer.write(results.plot_im)  # write the processed frame.

cap.release()
video_writer.release()
cv2.destroyAllWindows()  # destroy all opened windows

Ultralytics Solutions: ✅ {'source': None, 'model': 'yolo26n.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'show_boxes': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90.0, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 6, 'records': 5, 'fps': 24, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': True, 'iou': None, 'conf': 0.25, 'device': None, 'max_det': 300, 'quantize': None, 'imgsz': 640, 'tracker': 'tracktrack.yaml', 'verbose': True, 'data': 'images', 'recognizer': 'paddleocr'}
0: 640x640 0.4ms, 3 car
Speed: 245.1ms track, 0.4ms solution per image at shape (1, 3, 640, 640)

1: 640x640 0.4ms, 3 car
Speed: 121.6ms track, 0.4ms solution per image at shape (1, 3, 640, 640)

2: 640x640 0.3ms, 3 car
Speed: 126.3ms track, 0.3ms solution per image at shape (1, 3, 640,

## Limitations and Factors Affecting Accuracy

Speed estimation is an approximation, not a direct measurement from a vehicle's speedometer. Its accuracy can be affected by:

- **Camera calibration:** An unsuitable `meter_per_pixel` value leads to incorrect speed estimates.
- **Camera perspective:** Pixel distances may represent different real-world distances at different positions in the scene.
- **Object tracking:** Occlusion, crowded traffic, or ID switches can affect the movement history.
- **Video FPS:** An incorrect or unreliable frame rate affects elapsed-time calculations.
- **Video quality:** Motion blur, low resolution, and poor lighting can reduce detection and tracking quality.

For a reliable evaluation, compare estimated speeds with ground-truth measurements whenever possible.


## Conclusion

In this project, we explored how computer vision can be used to estimate the speed of moving objects in traffic videos. By combining YOLO-based object detection, object tracking, frame-rate information, and pixel-to-meter calibration, Ultralytics provides a practical speed-estimation workflow.

This type of solution can support traffic monitoring, road-safety analysis, and intelligent transportation systems.

### References
- [Ultralytics – Speed Estimation Guide](https://docs.ultralytics.com/guides/speed-estimation/)
- [Ultralytics – SpeedEstimator API Reference](https://docs.ultralytics.com/reference/solutions/speed_estimation/)
- [Ultralytics – Python Usage](https://docs.ultralytics.com/usage/python)

**Implementation note:** This notebook follows the current Ultralytics YOLO26 Speed Estimation guide. Review the official documentation if the API changes in a later release.


In [17]:
from google.colab import files
files.download("speed_management.avi")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>